# Exercise 1 — Smallest useful agent

## Build requirements

Python; minimal dependencies and no heavy orchestration framework.
Tools: lookup_document(query), calculate(expression), get_system_info().
A common model interface allowing local or hosted backends.

Validated schemas and safe execution limits.
Tool proposal, validator decision, result, latency, and provider-reported token usage logged.

## Acceptance criteria

- [] Agent selects an appropriate tool on at least 9 of 10 fixed test questions.

- [] Invalid tool arguments are rejected before execution.

- [] Tool failure produces understandable recovery or error response.

- [] Logs record proposals, validation, results, latency, and usage where available.

Switching underlying model does not require rewriting tool implementations.

Learning mode: Write the implementation personally. AI assistance can supply exercises, conceptual explanations, code review, and debugging hints; avoid auto-generating the entire solution.

In [1]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))

CUDA available: True
Device: NVIDIA GeForce RTX 4070 Laptop GPU
VRAM (GB): 8.59


In [2]:
from pathlib import Path
from dotenv import load_dotenv

# Locate this checkout, including when Jupyter starts in a chapter folder.
PROJECT_ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "pyproject.toml").is_file() and (p / "scratch_agents").is_dir()
)
import sys
# Use this checkout even if the selected kernel was installed elsewhere.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / ".env")

%load_ext autoreload
%autoreload 2

## Get System Info 

In [3]:
import os
import platform

def get_system_info() -> str:
    return (
        f"OS: {platform.system()}\n"
        f"Python version: {platform.python_version()}\n"
        f"CPU count: {os.cpu_count()}"
    )

print(get_system_info())

OS: Windows
Python version: 3.13.15
CPU count: 22


## lookup_document

In [4]:
documents = [
    "Refund rule: Returns are accepted within 30 days with a receipt.",
    "Office address: 123 Maple Street, Springfield.",
    "Product spec: The Trail camera has 24 megapixels and is waterproof.",
    "Shipping policy: Orders ship within 3 business days.",
    "Support hours: Monday through Friday, 9 AM to 5 PM."
]

def lookup_document(query: str) -> str:
    if len(query) > 100:
        return "Error: query must be 100 characters or fewer."

    words = query.lower().split()
    best_match = "No match found."
    best_score = 0

    for document in documents:
        score = sum(1 for word in words if word in document.lower())

        if score > best_score:
            best_match = document
            best_score = score

    return best_match[:200]

print(lookup_document("refund"))
print(lookup_document("office address"))
print(lookup_document("volcano"))


Refund rule: Returns are accepted within 30 days with a receipt.
Office address: 123 Maple Street, Springfield.
No match found.


## Calculate Expression

In [5]:
import ast
import operator

operations = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
}

def calculate(expression: str) -> str:
    if len(expression) > 100:
        return "Error: expression is too long."

    if any(char not in "0123456789.+-*/() " for char in expression):
        return "Error: only math expressions are allowed."

    def solve(node):
        if isinstance(node, ast.Constant):
            return node.value

        if isinstance(node, ast.UnaryOp):
            value = solve(node.operand)
            return -value if isinstance(node.op, ast.USub) else value

        return operations[type(node.op)](
            solve(node.left), solve(node.right)
        )

    try:
        tree = ast.parse(expression.strip(), mode="eval")

        allowed = (
            ast.Expression, ast.Constant, ast.BinOp, ast.UnaryOp,
            ast.Add, ast.Sub, ast.Mult, ast.Div, ast.UAdd, ast.USub
        )

        if any(not isinstance(node, allowed) for node in ast.walk(tree)):
            return "Error: unsupported expression."

        return str(solve(tree.body))

    except ZeroDivisionError:
        return "Error: division by zero."
    except (SyntaxError, ValueError, OverflowError):
        return "Error: invalid expression."


print(calculate("2 + 2"))
print(calculate("(10 - 4) / 2"))
print(calculate("import os"))
print(calculate("2 / 0"))

4
3.0
Error: only math expressions are allowed.
Error: division by zero.


## Tool definitions. 

In [6]:
calculator_tool_definition = {
    "type": "function",
    "function": {
        "name": "calculate",
        "description": (
            "Calculate a basic math expression using +, -, *, /, "
            "and parentheses. Exponentiation is not supported."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "expression": {
                    "type": "string",
                    "description": "Math expression, such as '(10 - 4) / 2'.",
                    "maxLength": 100
                }
            },
            "required": ["expression"]
        }
    }
}


In [7]:
lookup_document_tool_definition = {
    "type": "function",
    "function": {
        "name": "lookup_document",
        "description": "Search the sample documents for a matching snippet.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Keywords to search for.",
                    "maxLength": 100
                }
            },
            "required": ["query"]
        }
    }
}

get_system_info_tool_definition = {
    "type": "function",
    "function": {
        "name": "get_system_info",
        "description": "Get the computer's OS, Python version, and CPU count.",
        "parameters": {
            "type": "object",
            "properties": {},
            "required": []
        }
    }
}


In [8]:
tools = [
    calculator_tool_definition,
    lookup_document_tool_definition,
    get_system_info_tool_definition
]


### Validate 
- hello 

In [9]:
def validate_tool_call(name, arguments):
    expected_inputs = {
        "calculate": {"expression"},
        "lookup_document": {"query"},
        "get_system_info": set()
    }

    if not isinstance(name, str) or name not in expected_inputs:
        return False, "Unknown tool."

    if not isinstance(arguments, dict):
        return False, "Arguments must be a dictionary."

    if set(arguments) != expected_inputs[name]:
        return False, "Missing or unexpected arguments."

    for value in arguments.values():
        if not isinstance(value, str):
            return False, "Input must be a string."

        if len(value) > 100:
            return False, "Input must be 100 characters or fewer."

    return True, "Approved."


print(validate_tool_call("calculate", {"expression": "2 + 2"}))
print(validate_tool_call("calculate", {"expression": 42}))
print(validate_tool_call("delete_file", {}))


(True, 'Approved.')
(False, 'Input must be a string.')
(False, 'Unknown tool.')


In [10]:
import json
from litellm import completion

tools = tools = [
    calculator_tool_definition,
    lookup_document_tool_definition,
    get_system_info_tool_definition
]

response_without_tool = completion(
    model='gpt-5.4-mini',
    messages=[{"role": "user", "content": "What is the capital of South Korea?"}],
    tools=tools,
)
print("# Capital question doesn't need a tool call")
print(response_without_tool.choices[0].message.content)
print(response_without_tool.choices[0].message.tool_calls)

response_with_tool = completion(
    model='gpt-5.4-mini',
    messages=[{"role": "user", "content": "What is 1234 x 5678?"}],
    tools=tools,
)
print("\n# Multiplication question needs a tool call")
print(response_with_tool.choices[0].message.content)
print(response_with_tool.choices[0].message.tool_calls)

# Capital question doesn't need a tool call
The capital of South Korea is **Seoul**.
None

# Multiplication question needs a tool call
None
[ChatCompletionMessageToolCall(function=Function(arguments='{"expression":"1234 * 5678"}', name='calculate'), id='call_6CvCyRSFRznlQ89rzcRZbxY9', type='function')]


In [24]:
from copy import deepcopy

bad_call = deepcopy(response_with_tool.choices[0].message.tool_calls[0])
bad_call.id = "test_wrong_input_type"
bad_call.function.arguments = '{"expression": 42}'

tool_calls_to_run = [bad_call]


In [40]:
from copy import deepcopy

malformed_call = deepcopy(
    response_with_tool.choices[0].message.tool_calls[0]
)
malformed_call.id = "test_invalid_json"
malformed_call.function.arguments = '{"expression":'

tool_calls_to_run = [malformed_call]


In [41]:
import json
from time import perf_counter

available_tools = {
    "calculate": calculate,
    "lookup_document": lookup_document,
    "get_system_info": get_system_info
}

for tool_call in tool_calls_to_run:
    name = tool_call.function.name

    try:
        arguments = json.loads(tool_call.function.arguments)
    except (json.JSONDecodeError, TypeError, RecursionError):
        logs.append({
            "stage": "tool_execution",
            "tool_call_id": tool_call.id,
            "tool": name,
            "raw_arguments": tool_call.function.arguments,
            "approved": False,
            "reason": "Invalid JSON arguments.",
            "result": None,
            "tool_execution_seconds": None
        })
        print("Rejected: invalid JSON arguments.")
        continue


    approved, reason = validate_tool_call(name, arguments)
    print(f"{name}: {reason}")

    if not approved:
        logs.append({
            "stage": "tool_execution",
            "tool_call_id": tool_call.id,
            "tool": name,
            "arguments": arguments,
            "approved": approved,
            "reason": reason,
            "result": None,
            "tool_execution_seconds": None
        })
        continue

    start = perf_counter()
    result = available_tools[name](**arguments)
    elapsed = perf_counter() - start

    logs.append({
        "stage": "tool_execution",
        "tool_call_id": tool_call.id,
        "tool": name,
        "arguments": arguments,
        "approved": approved,
        "reason": reason,
        "result": result,
        "tool_execution_seconds": elapsed
    })

    print("Result:", result)
    print(f"Tool execution: {elapsed:.6f} seconds")


Rejected: invalid JSON arguments.


In [ ]:
failure_message = failure_response.choices[0].message
failure_call = failure_message.tool_calls[0]

failure_messages = [
    {"role": "user", "content": failure_question},
    failure_message.model_dump(exclude_none=True),
    {
        "role": "tool",
        "tool_call_id": failure_call.id,
        "content": str(result)
    }
]

failure_final_response = completion(
    model="gpt-5.4-mini",
    messages=failure_messages
)

print(failure_final_response.choices[0].message.content)



Division by zero is undefined, so \(2 / 0\) cannot be evaluated.


In [42]:
print(json.dumps(logs[-1], indent=2))


{
  "stage": "tool_execution",
  "tool_call_id": "test_invalid_json",
  "tool": "calculate",
  "raw_arguments": "{\"expression\":",
  "approved": false,
  "reason": "Invalid JSON arguments.",
  "result": null,
  "tool_execution_seconds": null
}


In [28]:
tool_calls_to_run = response_with_tool.choices[0].message.tool_calls or []


In [35]:
failure_question = "Use the calculator to evaluate 2 / 0."

failure_response = completion(
    model="gpt-5.4-mini",
    messages=[{"role": "user", "content": failure_question}],
    tools=tools,
    tool_choice={"type": "function", "function": {"name": "calculate"}}
)

tool_calls_to_run = failure_response.choices[0].message.tool_calls or []

print(failure_response.choices[0].message.tool_calls)


[ChatCompletionMessageToolCall(function=Function(arguments='{"expression":"2 / 0"}', name='calculate'), id='call_Ge6003IZomeOoV0RR2uzEAzJ', type='function')]


In [32]:
assistant_message = response_with_tool.choices[0].message
tool_call = assistant_message.tool_calls[0]

messages = [
    {"role": "user", "content": "What is 1234 x 5678?"},
    assistant_message.model_dump(exclude_none=True),
    {
        "role": "tool",
        "tool_call_id": tool_call.id,
        "content": str(result)
    }
]

final_response = completion(
    model="gpt-5.4-mini",
    messages=messages
)

print(final_response.choices[0].message.content)


1234 × 5678 = **7,006,652**


In [34]:
logs = []

for stage, response in [
    ("tool_proposal", response_with_tool),
    ("final_answer", final_response)
]:
    usage = getattr(response, "usage", None)

    logs.append({
        "stage": stage,
        "model": response.model,
        "token_usage": usage.model_dump() if usage is not None else None
    })

print(json.dumps(logs, indent=2))



[
  {
    "stage": "tool_proposal",
    "model": "gpt-5.4-mini-2026-03-17",
    "token_usage": {
      "completion_tokens": 21,
      "prompt_tokens": 230,
      "total_tokens": 251,
      "completion_tokens_details": {
        "accepted_prediction_tokens": 0,
        "audio_tokens": 0,
        "reasoning_tokens": 0,
        "rejected_prediction_tokens": 0,
        "text_tokens": null,
        "image_tokens": null,
        "video_tokens": null
      },
      "prompt_tokens_details": {
        "audio_tokens": 0,
        "cached_tokens": 0,
        "text_tokens": null,
        "image_tokens": null,
        "video_tokens": null
      }
    }
  },
  {
    "stage": "final_answer",
    "model": "gpt-5.4-mini-2026-03-17",
    "token_usage": {
      "completion_tokens": 17,
      "prompt_tokens": 50,
      "total_tokens": 67,
      "completion_tokens_details": {
        "accepted_prediction_tokens": 0,
        "audio_tokens": 0,
        "reasoning_tokens": 0,
        "rejected_prediction_token

In [38]:
test_questions = [
    ("What is 1234 multiplied by 5678?", "calculate"),
    ("What is (875 + 129) / 4?", "calculate"),
    ("What is 9876 minus 4321?", "calculate"),
    ("What is our refund rule?", "lookup_document"),
    ("What is our office address?", "lookup_document"),
    ("What are the specifications of the Trail camera?", "lookup_document"),
    ("What operating system is this Python program running on?", "get_system_info"),
    ("What Python version and CPU count does this computer have?", "get_system_info"),
    ("What is the capital of South Korea?", None),
    ("Write a friendly one-sentence greeting.", None)
]


In [39]:
selection_results = []

for number, (question, expected) in enumerate(test_questions, start=1):
    start = perf_counter()

    response = completion(
        model="gpt-5.4-mini",
        messages=[{"role": "user", "content": question}],
        tools=tools,
        tool_choice="auto"
    )

    elapsed = perf_counter() - start
    message = response.choices[0].message
    calls = message.tool_calls or []

    selected = [call.function.name for call in calls]
    expected_tools = [] if expected is None else [expected]
    passed = selected == expected_tools

    usage = getattr(response, "usage", None)

    selection_results.append({
        "question": question,
        "expected": expected_tools,
        "selected": selected,
        "passed": passed,
        "proposals": [call.model_dump() for call in calls],
        "answer": message.content,
        "model": response.model,
        "model_call_seconds": elapsed,
        "token_usage": usage.model_dump() if usage is not None else None
    })

    status = "PASS" if passed else "FAIL"
    print(f"{number}. {status} | expected={expected_tools} | selected={selected}")

score = sum(item["passed"] for item in selection_results)

print(f"\nTool-selection score: {score}/{len(test_questions)}")
print("Acceptance check:", "PASS" if score >= 9 else "FAIL")


1. PASS | expected=['calculate'] | selected=['calculate']
2. PASS | expected=['calculate'] | selected=['calculate']
3. PASS | expected=['calculate'] | selected=['calculate']
4. PASS | expected=['lookup_document'] | selected=['lookup_document']
5. PASS | expected=['lookup_document'] | selected=['lookup_document']
6. PASS | expected=['lookup_document'] | selected=['lookup_document']
7. PASS | expected=['get_system_info'] | selected=['get_system_info']
8. PASS | expected=['get_system_info'] | selected=['get_system_info']
9. PASS | expected=[] | selected=[]
10. PASS | expected=[] | selected=[]

Tool-selection score: 10/10
Acceptance check: PASS
